<a href="https://colab.research.google.com/github/CHENYIZHAO1203/DSSS/blob/main/notebooks/W02_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Lab

**Datasets**:

The following datasets are used in this lab.

- [nyc_subway_stations.tsv](https://storage.googleapis.com/qm2/CASA0025/nyc_subway_stations.tsv)
- [nyc_neighborhoods.tsv](https://storage.googleapis.com/qm2/CASA0025/nyc_neighborhoods.tsv)

In [1]:
%pip install duckdb duckdb-engine jupysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 kB 13.5 MB/s eta 0:00:00


In [2]:
import duckdb

%load_ext sql

In [3]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

## Question 1: Creating Tables

Create a database, then write a SQL query to create a table named `nyc_subway_stations` and load the data from the file `nyc_subway_stations.tsv` into it. Similarly, create a table named `nyc_neighborhoods` and load the data from the file `nyc_neighborhoods.tsv` into it.

In [8]:
%sql duckdb:///:memory:
# %sql duckdb:///path/to/file.db

In [9]:
%%sql
INSTALL httpfs;
LOAD httpfs;

,Success


In [10]:
%%sql
SELECT * FROM duckdb_extensions();

,extension_name,loaded,installed,install_path,description,aliases,extension_version,install_mode,installed_from
0,autocomplete,False,False,,Adds support for autocomplete in the shell,[],,NOT_INSTALLED,
1,aws,False,False,,Provides features that depend on the AWS SDK,[],,NOT_INSTALLED,
2,azure,False,False,,Adds a filesystem abstraction for Azure blob s...,[],,NOT_INSTALLED,
3,core_functions,True,True,(BUILT-IN),Core function library,[],,STATICALLY_LINKED,
4,delta,False,False,,Adds support for Delta Lake,[],,NOT_INSTALLED,
5,ducklake,False,False,,"Adds support for DuckLake, SQL as a Lakehouse ...",[],,NOT_INSTALLED,
6,encodings,False,False,,All unicode encodings to UTF-8,[],,NOT_INSTALLED,
7,excel,False,False,,Adds support for Excel-like format strings,[],,NOT_INSTALLED,
8,fts,False,False,,Adds support for Full-Text Search Indexes,[],,NOT_INSTALLED,
9,httpfs,True,True,/root/.duckdb/extensions/v1.3.2/linux_amd64/ht...,Adds support for reading and writing files ove...,"[http, https, s3]",af7bcaf,REPOSITORY,core


In [11]:
%%sql
SELECT * FROM "https://storage.googleapis.com/qm2/CASA0025/nyc_subway_stations.tsv"

,OBJECTID,ID,NAME,ALT_NAME,CROSS_ST,LONG_NAME,LABEL,BOROUGH,NGHBHD,ROUTES,TRANSFERS,COLOR,EXPRESS,CLOSED
0,1,376.0,Cortlandt St,None,Church St,"Cortlandt St (R,W) Manhattan","Cortlandt St (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,<NA>
1,2,2.0,Rector St,None,None,Rector St (1) Manhattan,Rector St (1),Manhattan,None,1,1,RED,None,<NA>
2,3,1.0,South Ferry,None,None,South Ferry (1) Manhattan,South Ferry (1),Manhattan,None,1,1,RED,None,<NA>
3,4,125.0,138th St,Grand Concourse,Grand Concourse,"138th St / Grand Concourse (4,5) Bronx","138th St / Grand Concourse (4,5)",Bronx,None,"4,5","4,5",GREEN,None,<NA>
4,5,126.0,149th St,Grand Concourse,Grand Concourse,149th St / Grand Concourse (4) Bronx,149th St / Grand Concourse (4),Bronx,None,4,"2,4,5",GREEN,express,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,487,909.0,JFK Terminal 8,None,None,"JFK Terminal 8, Queens",JFK Terminal 8,Queens,None,None,None,AIR-BLUE,None,<NA>
487,488,903.0,Federal Circle,Rental Car,None,"Federal Circle / Rental Car, Queens",Federal Circle / Rental Car,Queens,None,None,None,AIR-BLUE,None,<NA>
488,489,902.0,Long Term Parking,None,None,"Long Term Parking, Queens",Long Term Parking,Queens,None,None,None,AIR-BLUE,None,<NA>
489,490,901.0,Howard Beach,None,159th Ave,"Howard Beach, Queens",Howard Beach,Queens,None,None,A,AIR-BLUE,None,<NA>


In [12]:
%%sql
SELECT * FROM "https://storage.googleapis.com/qm2/CASA0025/nyc_neighborhoods.tsv";

,BORONAME,NAME
0,Brooklyn,Bensonhurst
1,Manhattan,East Village
2,Manhattan,West Village
3,The Bronx,Throggs Neck
4,The Bronx,Wakefield-Williamsbridge
...,...,...
124,Brooklyn,Red Hook
125,Queens,Douglastown-Little Neck
126,Queens,Whitestone
127,Queens,Steinway


In [13]:
%%sql

CREATE TABLE nyc_subway_stations AS SELECT * FROM "https://storage.googleapis.com/qm2/CASA0025/nyc_subway_stations.tsv";

,Success


In [14]:
%%sql

CREATE TABLE nyc_neighborhoods AS SELECT * FROM "https://storage.googleapis.com/qm2/CASA0025/nyc_neighborhoods.tsv";

,Success


from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['index']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'index'}, axis=1)
              .sort_values('index', ascending=True))
  xs = counted['index']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_0.sort_values('index', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('count()')

In [15]:
%%sql

FROM nyc_subway_stations;

,OBJECTID,ID,NAME,ALT_NAME,CROSS_ST,LONG_NAME,LABEL,BOROUGH,NGHBHD,ROUTES,TRANSFERS,COLOR,EXPRESS,CLOSED
0,1,376.0,Cortlandt St,None,Church St,"Cortlandt St (R,W) Manhattan","Cortlandt St (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,<NA>
1,2,2.0,Rector St,None,None,Rector St (1) Manhattan,Rector St (1),Manhattan,None,1,1,RED,None,<NA>
2,3,1.0,South Ferry,None,None,South Ferry (1) Manhattan,South Ferry (1),Manhattan,None,1,1,RED,None,<NA>
3,4,125.0,138th St,Grand Concourse,Grand Concourse,"138th St / Grand Concourse (4,5) Bronx","138th St / Grand Concourse (4,5)",Bronx,None,"4,5","4,5",GREEN,None,<NA>
4,5,126.0,149th St,Grand Concourse,Grand Concourse,149th St / Grand Concourse (4) Bronx,149th St / Grand Concourse (4),Bronx,None,4,"2,4,5",GREEN,express,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,487,909.0,JFK Terminal 8,None,None,"JFK Terminal 8, Queens",JFK Terminal 8,Queens,None,None,None,AIR-BLUE,None,<NA>
487,488,903.0,Federal Circle,Rental Car,None,"Federal Circle / Rental Car, Queens",Federal Circle / Rental Car,Queens,None,None,None,AIR-BLUE,None,<NA>
488,489,902.0,Long Term Parking,None,None,"Long Term Parking, Queens",Long Term Parking,Queens,None,None,None,AIR-BLUE,None,<NA>
489,490,901.0,Howard Beach,None,159th Ave,"Howard Beach, Queens",Howard Beach,Queens,None,None,A,AIR-BLUE,None,<NA>


In [16]:
%%sql

FROM nyc_neighborhoods;

,BORONAME,NAME
0,Brooklyn,Bensonhurst
1,Manhattan,East Village
2,Manhattan,West Village
3,The Bronx,Throggs Neck
4,The Bronx,Wakefield-Williamsbridge
...,...,...
124,Brooklyn,Red Hook
125,Queens,Douglastown-Little Neck
126,Queens,Whitestone
127,Queens,Steinway


## Question 2: Column Filtering

Write a SQL query to display the `ID`, `NAME`, and `BOROUGH` of each subway station in the `nyc_subway_stations` dataset.

In [17]:
%%sql

SELECT * FROM nyc_subway_stations;

,OBJECTID,ID,NAME,ALT_NAME,CROSS_ST,LONG_NAME,LABEL,BOROUGH,NGHBHD,ROUTES,TRANSFERS,COLOR,EXPRESS,CLOSED
0,1,376.0,Cortlandt St,None,Church St,"Cortlandt St (R,W) Manhattan","Cortlandt St (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,<NA>
1,2,2.0,Rector St,None,None,Rector St (1) Manhattan,Rector St (1),Manhattan,None,1,1,RED,None,<NA>
2,3,1.0,South Ferry,None,None,South Ferry (1) Manhattan,South Ferry (1),Manhattan,None,1,1,RED,None,<NA>
3,4,125.0,138th St,Grand Concourse,Grand Concourse,"138th St / Grand Concourse (4,5) Bronx","138th St / Grand Concourse (4,5)",Bronx,None,"4,5","4,5",GREEN,None,<NA>
4,5,126.0,149th St,Grand Concourse,Grand Concourse,149th St / Grand Concourse (4) Bronx,149th St / Grand Concourse (4),Bronx,None,4,"2,4,5",GREEN,express,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,487,909.0,JFK Terminal 8,None,None,"JFK Terminal 8, Queens",JFK Terminal 8,Queens,None,None,None,AIR-BLUE,None,<NA>
487,488,903.0,Federal Circle,Rental Car,None,"Federal Circle / Rental Car, Queens",Federal Circle / Rental Car,Queens,None,None,None,AIR-BLUE,None,<NA>
488,489,902.0,Long Term Parking,None,None,"Long Term Parking, Queens",Long Term Parking,Queens,None,None,None,AIR-BLUE,None,<NA>
489,490,901.0,Howard Beach,None,159th Ave,"Howard Beach, Queens",Howard Beach,Queens,None,None,A,AIR-BLUE,None,<NA>


In [18]:
%%sql
SELECT
  ID,
  NAME,
  BOROUGH
FROM nyc_subway_stations;

,ID,NAME,BOROUGH
0,376.0,Cortlandt St,Manhattan
1,2.0,Rector St,Manhattan
2,1.0,South Ferry,Manhattan
3,125.0,138th St,Bronx
4,126.0,149th St,Bronx
...,...,...,...
486,909.0,JFK Terminal 8,Queens
487,903.0,Federal Circle,Queens
488,902.0,Long Term Parking,Queens
489,901.0,Howard Beach,Queens


## Question 3: Row Filtering

Write a SQL query to find all subway stations in the `nyc_subway_stations` dataset that are located in the borough of Manhattan.

In [19]:
%%sql
SELECT *
FROM nyc_subway_stations
WHERE BOROUGH = 'Manhattan';

,OBJECTID,ID,NAME,ALT_NAME,CROSS_ST,LONG_NAME,LABEL,BOROUGH,NGHBHD,ROUTES,TRANSFERS,COLOR,EXPRESS,CLOSED
0,1,376.0,Cortlandt St,None,Church St,"Cortlandt St (R,W) Manhattan","Cortlandt St (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,<NA>
1,2,2.0,Rector St,None,None,Rector St (1) Manhattan,Rector St (1),Manhattan,None,1,1,RED,None,<NA>
2,3,1.0,South Ferry,None,None,South Ferry (1) Manhattan,South Ferry (1),Manhattan,None,1,1,RED,None,<NA>
3,25,296.0,W 4th St,None,6th Ave,"W 4th St (B,D,F,V) Manhattan","W 4th St (B,D,F,V)",Manhattan,None,"B,D,F,V","A,B,C,D,E,F,V",ORANGE,express,<NA>
4,242,121.0,103rd St,None,Lexington Ave,103rd St (6) Manhattan,103rd St (6),Manhattan,None,6,6,GREEN,None,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,374,72.0,Wall St,None,William St,"Wall St (2,3) Manhattan","Wall St (2,3)",Manhattan,None,"2,3","2,3",RED,express,<NA>
136,375,378.0,Whitehall St,South Ferry,Water St,"Whitehall St / South Ferry (R,W) Manhattan","Whitehall St / South Ferry (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,<NA>
137,383,491.0,Cortlandt St,None,West Broadway,Cortlandt St (1) Manhattan,Cortlandt St (1),Manhattan,None,1,1,CLOSED,None,True
138,480,492.0,World Trade Center,Chambers St,Church St,World Trade Center / Chambers St (E) Manhattan,World Trade Center / Chambers St (E),Manhattan,None,E,"A,C,E,2,3",BLUE,None,<NA>


## Question 4: Sorting Results

Write a SQL query to list the subway stations in the `nyc_subway_stations` dataset in alphabetical order by their names.

In [20]:
%%sql
SELECT *
FROM nyc_subway_stations
ORDER BY NAME ASC;

,OBJECTID,ID,NAME,ALT_NAME,CROSS_ST,LONG_NAME,LABEL,BOROUGH,NGHBHD,ROUTES,TRANSFERS,COLOR,EXPRESS,CLOSED
0,244,22.0,103rd St,None,Broadway,103rd St (1) Manhattan,103rd St (1),Manhattan,None,1,1,RED,None,<NA>
1,243,192.0,103rd St,None,Central Park West,"103rd St (A,B,C) Manhattan","103rd St (A,B,C)",Manhattan,None,"A,B,C","A,B,C",BLUE-ORANGE,None,<NA>
2,242,121.0,103rd St,None,Lexington Ave,103rd St (6) Manhattan,103rd St (6),Manhattan,None,6,6,GREEN,None,<NA>
3,376,174.0,103rd St,Corona Plaza,Roosevelt Ave,103rd St / Corona Plaza (7) Queens,103rd St / Corona Plaza (7),Queens,None,7,7,PURPLE,None,<NA>
4,377,239.0,104th St,None,Liberty Ave,104th St (A) Queens,104th St (A),Queens,None,A,A,BLUE,None,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,72,138.0,Woodlawn,None,Jerome Ave,Woodlawn (4) Bronx,Woodlawn (4),Bronx,None,4,4,GREEN,None,<NA>
487,480,492.0,World Trade Center,Chambers St,Church St,World Trade Center / Chambers St (E) Manhattan,World Trade Center / Chambers St (E),Manhattan,None,E,"A,C,E,2,3",BLUE,None,<NA>
488,240,334.0,Wyckoff Ave,Myrtle Ave,Myrtle Ave,Wyckoff Ave / Myrtle Ave (M) Brooklyn,Wyckoff Ave / Myrtle Ave (M),Brooklyn,None,M,"L,M",BROWN,None,<NA>
489,241,303.0,York St,None,Jay St,York St (F) Brooklyn,York St (F),Brooklyn,None,F,F,ORANGE,None,<NA>


## Question 5: Unique Values

Write a SQL query to find the distinct boroughs represented in the `nyc_subway_stations` dataset.

In [ ]:
# Add your code here.

## Question 6: Counting Rows

Write a SQL query to count the number of subway stations in each borough in the `nyc_subway_stations` dataset.

In [ ]:
# Add your code here.

## Question 7: Aggregating Data

Write a SQL query to list the number of subway stations in each borough, sorted in descending order by the count.

In [ ]:
# Add your code here.

## Question 8: Joining Tables

Write a SQL query to join the `nyc_subway_stations` and `nyc_neighborhoods` datasets on the borough name, displaying the subway station name and the neighborhood name.

In [ ]:
# Add your code here.

## Question 9: String Manipulation

Write a SQL query to display the names of subway stations in the `nyc_subway_stations` dataset that contain the word "St" in their names.

In [ ]:
# Add your code here.

## Question 10: Filtering with Multiple Conditions

Write a SQL query to find all subway stations in the `nyc_subway_stations` dataset that are in the borough of Brooklyn and have routes that include the letter "R".

In [ ]:
# Add your code here.

Once you've completed your attempt, you can check your answers [here](https://github.com/oballinger/CASA0025/blob/main/notebooks/W02_lab_solution.ipynb).